# Классификация токсичных комментариев

Чистый воспроизводимый ноутбук к финальному проекту. Здесь используется независимый test split, а модель и порог выбираются только по validation.

## 1. Установка

Из корня репозитория выполните `python -m pip install -r requirements-dev.txt`. Данные загрузятся автоматически при первом запуске.

In [ ]:
from pathlib import Path

import json
import joblib
import matplotlib.pyplot as plt
import pandas as pd

from toxic_comments import load_splits, run_classical_experiment

RANDOM_STATE = 42
ARTIFACTS_DIR = Path("../artifacts-notebook")

## 2. Данные

Используется `mteb/toxic_conversations_50k`: 50 000 train- и 50 000 test-комментариев Civil Comments. Для быстрого учебного запуска ниже берётся стратифицированная подвыборка; для итогового результата используйте CLI без `sample_size`.

In [ ]:
splits = load_splits(
    cache_dir=Path("../data"),
    random_state=RANDOM_STATE,
    sample_size=5000,
)

summary = pd.DataFrame(
    {
        name: {
            "rows": len(frame),
            "toxic": int(frame["label"].sum()),
            "toxic_share": frame["label"].mean(),
        }
        for name, frame in {
            "train": splits.train,
            "validation": splits.validation,
            "test": splits.test,
        }.items()
    }
).T
summary

In [ ]:
summary["toxic_share"].plot(kind="bar", ylim=(0, 0.12), title="Доля токсичных комментариев")
plt.ylabel("share")
plt.show()

## 3. Обучение и оценка

Сравниваются Dummy baseline, word TF-IDF + Logistic Regression и word+character TF-IDF + Linear SVM. Порог каждой модели максимизирует F1 токсичного класса на validation.

In [ ]:
result = run_classical_experiment(
    splits=splits,
    artifacts_dir=ARTIFACTS_DIR,
    max_features=10_000,
    random_state=RANDOM_STATE,
)
result.metrics.round(4)

In [ ]:
result.metrics.pivot(index="model", columns="split", values="f1_toxic").plot(
    kind="bar",
    title="F1 токсичного класса",
)
plt.ylim(0, 1)
plt.ylabel("F1")
plt.show()

## 4. Инференс

Для решения используется сохранённый validation-порог, а не фиксированные 0.5.

In [ ]:
model = joblib.load(result.model_path)
metadata = json.loads((ARTIFACTS_DIR / "model_metadata.json").read_text())

examples = [
    "Thank you for the clear explanation.",
    "You are an idiot and nobody wants you here.",
]
scores = model.predict_proba(examples)[:, 1]
pd.DataFrame(
    {
        "text": examples,
        "score": scores,
        "prediction": (scores >= metadata["threshold"]).astype(int),
    }
)

## 5. Выводы и ограничения

Для несбалансированной задачи необходимо анализировать F1 и PR-AUC, а не только accuracy. Числа этого эксперимента нельзя напрямую сравнивать с публикациями, использующими другие split, labels или fairness-метрики. Модель может наследовать identity bias Civil Comments, поэтому для реального применения нужны subgroup-анализ и ручная модерация.